In [0]:
# Path to your uploaded files
sales_path = "/Volumes/retail_project/bronze/raw_files/train.csv"
store_path = "/Volumes/retail_project/bronze/raw_files/store.csv"

# Read train.csv (historical sales)
df_sales_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(sales_path)
)

display(df_sales_raw)

In [0]:
df_store_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(store_path)
)

display(df_store_raw)

In [0]:
print("Sales rows:", df_sales_raw.count())
print("Sales columns:", df_sales_raw.columns)
print()
print("Store rows:", df_store_raw.count())
print("Store columns:", df_store_raw.columns)

In [0]:
# Write sales data as Bronze Delta table
df_sales_raw.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.bronze.sales_raw"
)

# Write store metadata as Bronze Delta table
df_store_raw.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.bronze.store_raw"
)

print("Bronze tables written successfully.")

In [0]:
display(spark.sql("SELECT * FROM retail_project.bronze.sales_raw LIMIT 10"))
display(spark.sql("SELECT * FROM retail_project.bronze.store_raw LIMIT 10"))

In [0]:
# Check for nulls in key columns - useful to document for the Silver cleaning step
from pyspark.sql.functions import col, sum as spark_sum

null_counts_store = df_store_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_store_raw.columns
])
display(null_counts_store)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts_sales = df_sales_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_sales_raw.columns
])
display(null_counts_sales)